# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Name: {getattr(metadata, 'name', None)}\nDescription: {getattr(metadata, 'description', None)}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll inspect the record sets, fields, and columns. All entities are referenced by their `@id`.

In [ ]:
from pprint import pprint

# List all record sets and their fields by @id
record_sets = list(dataset.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")
for record_set in record_sets:
    print(f"Record set @id: {getattr(record_set, '@id', None)}")
    print(f"  Name: {getattr(record_set, 'name', None)}\n  Description: {getattr(record_set, 'description', None)}")
    print("  Fields (by @id):")
    for field in getattr(record_set, 'fields', []):
        print(f"    - {getattr(field, '@id', None)} (column: {getattr(field, 'column', None)})")
    print()
if not record_sets:
    print("No record_sets found in metadata. This dataset may be comprised of file-based entities, so we will check distributions and data files instead.")

# Print distribution and file objects
distributions = getattr(metadata, 'distribution', [])
if distributions:
    print("Dataset distributions (files):")
    for dist in distributions:
        print(dist)
else:
    print("No distributions found in metadata.")

## 3. Data Extraction
Load data from a specific record set or file object into a DataFrame for analysis. Use the `@id`s from the overview above.

In [ ]:
# If there are no explicit record sets, inspect data via file objects or all records
dataframes = {}

# First, try to use record sets if present
if record_sets:
    # Suppose the first record set is primary, and we extract its @id
    primary_record_set = record_sets[0]
    record_set_id = getattr(primary_record_set, '@id', None)
    print(f"Extracting records from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

    print(f"Available columns in this record set:")
    print(dataframes[record_set_id].columns.tolist())
    display(dataframes[record_set_id].head())
else:
    # If no record sets, try to extract all records generically
    print("Extracting records from unnamed file object(s)...")
    try:
        records = list(dataset.records())
        df = pd.DataFrame(records)
        dataframes['main'] = df
        print(f"Extracted {len(df)} records.")
        print("Columns:", df.columns.tolist())
        display(df.head())
    except Exception as e:
        print(f"Unable to extract records: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

You may need to adapt column names and field @ids according to the actual loaded DataFrame.

In [ ]:
import numpy as np

# For demo purposes, let's choose plausible numeric fields
df = None
if record_sets:
    df = dataframes[record_set_id]
else:
    df = dataframes.get('main', pd.DataFrame())

# Display available columns to help users adjust field selection
print("Available columns:", df.columns.tolist())

# Suppose 'MW' (molecular weight) is present as a numeric field (or adapt as found)
numeric_field_id = None
for col in df.columns:
    if col.lower() in ['mw', 'molecular_weight', 'molecularweight'] or 'weight' in col.lower():
        numeric_field_id = col
        break
if numeric_field_id is None:
    # Fallback to generic float/integer columns
    for col in df.columns:
        if np.issubdtype(df[col].dtype, np.number):
            numeric_field_id = col
            break
print(f"Using numeric field: {numeric_field_id}")

if numeric_field_id:
    # Filter and normalize
    threshold = np.nanmean(df[numeric_field_id])
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Attempt to group by a possible categorical field
    group_field_candidates = [col for col in df.columns if col.lower() in ['description', 'category', 'accession'] or df[col].dtype == object]
    group_field = group_field_candidates[0] if group_field_candidates else None
    if group_field and group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field}:")
        display(grouped_df.head())
else:
    print("No numeric field found for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field
if numeric_field_id:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.show()

    # Boxplot per group field if present
    if group_field and group_field in df.columns:
        plt.figure(figsize=(12,6))
        top_groups = df[group_field].value_counts().nlargest(10).index
        sns.boxplot(x=group_field, y=numeric_field_id, data=df[df[group_field].isin(top_groups)])
        plt.xticks(rotation=45)
        plt.title(f'{numeric_field_id} by {group_field} (Top 10 Categories)')
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
In this notebook, we loaded the Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells dataset using the `mlcroissant` library, explored its structure, extracted and analyzed core fields, and visualized distributions. All data references (record sets, fields) were handled via their `@id`. This workflow can be adapted for additional EDA or downstream ML tasks.